# Week 2 ,Notebook A: prepare the data



In [19]:
# Importing library and explicit paths
import re
from pathlib import Path
import numpy as np
import pandas as pd

#stating location of files in my local device
PROJECT_DIR = Path.home() / "Downloads" / "main" / "Week_2"

DATA_DIR = PROJECT_DIR / "data"

WEEK1_PATH = PROJECT_DIR.parent / "Week_1" / "data" / "processed" / "indic_sentiment_processed.csv"
MANIFEST_PATH = DATA_DIR / "split_manifest.csv"
INTENT_LABELS_PATH = DATA_DIR / "intent_labels.csv"
DATASET_PATH = DATA_DIR / "week2_dataset.csv"      

In [20]:
#Loadinh the Week 1 data, the split manifest and the intent labels
week1 = pd.read_csv(WEEK1_PATH)
manifest = pd.read_csv(MANIFEST_PATH)
labels = pd.read_csv(INTENT_LABELS_PATH)

required_columns = {
    "Week 1 file": (week1, ["language", "split", "text", "label"]),
    "split_manifest.csv": (manifest, ["review_id", "split", "english", "label", "label_id", "part"]),
    "intent_labels.csv": (labels, ["review_id", "intent", "unsure"]),
}
for name, (table, columns) in required_columns.items():
    missing = [c for c in columns if c not in table.columns]
    if missing:
        raise ValueError(f"{name} is missing the column(s) {missing}. It has: {list(table.columns)}")
    print(f"{name}: {len(table)} rows")

Week 1 file: 3468 rows
split_manifest.csv: 1152 rows
intent_labels.csv: 1152 rows


In [14]:
# Week 1: languages, texts, review ids, one sentiment label per review
if set(week1["language"]) != {"hi", "bn", "mr"}:
    raise ValueError(f"Week 1 must contain hi, bn and mr but has {sorted(set(week1['language']))}")
if week1["text"].isna().any() or week1["text"].str.strip().eq("").any():
    raise ValueError("Week 1 contains empty review text")

# the id is built from the row order inside each language and split
week1["review_id"] = week1["split"].astype(str) + "_" + week1.groupby(["language", "split"]).cumcount().astype(str)
ids = {lang: set(group["review_id"]) for lang, group in week1.groupby("language")}
if not (ids["hi"] == ids["bn"] == ids["mr"]):
    raise ValueError("hi, bn and mr do not have the same review ids: the languages are not row-aligned")

sentiment = week1.pivot(index="review_id", columns="language", values="label")
texts = week1.pivot(index="review_id", columns="language", values="text")
if sentiment.nunique(axis=1, dropna=False).ne(1).any():      # a missing label in only one language also counts as a difference
    raise ValueError("the sentiment label of some reviews differs between languages")
week1_label = sentiment["hi"]

# manifest and intent labels
for name, table in [("split_manifest.csv", manifest), ("intent_labels.csv", labels)]:
    if table["review_id"].isna().any() or table["review_id"].duplicated().any():
        raise ValueError(f"{name}: review_id must be present and unique")
if set(manifest["review_id"]) != set(labels["review_id"]):
    raise ValueError("split_manifest.csv and intent_labels.csv do not contain the same review ids")
if set(manifest["part"]) != {"train", "val", "test"}:
    raise ValueError(f"'part' must be train / val / test but has {sorted(set(manifest['part']))}")

allowed_intents = ["Inquiry", "Opinion", "Factual"]          # a list, not a set: the order gives the class numbers
if labels["intent"].isna().any() or not set(labels["intent"]) <= set(allowed_intents):
    raise ValueError(f"every review needs one intent from {allowed_intents}")

# the manifest must match Week 1: ids, sentiment labels and the original split
not_in_week1 = sorted(set(manifest["review_id"]) - ids["hi"])
if not_in_week1:
    raise ValueError(f"{len(not_in_week1)} manifest review_ids are not in Week 1, e.g. {not_in_week1[:5]}")
manifest_label = manifest.set_index("review_id")["label"]
if (manifest_label != week1_label.reindex(manifest_label.index)).any():
    raise ValueError("the sentiment label of some reviews differs between the manifest and Week 1")
if manifest["split"].ne(manifest["review_id"].str.partition("_")[0]).any():
    raise ValueError("'split' in the manifest does not match the review_id for some reviews")
print("Data validation passed.")

Data validation passed.


In [15]:
# Week 1 reviews that are not in the manifest: each one needs a reason
kept_texts = set(map(tuple, texts.loc[manifest["review_id"]].values))
excluded = []
for review in sorted(ids["hi"] - set(manifest["review_id"])):
    if pd.isna(week1_label[review]):
        excluded.append((review, "missing sentiment label"))
    elif tuple(texts.loc[review]) in kept_texts:
        excluded.append((review, "repeated review (same text in hi, bn and mr)"))
    else:
        raise ValueError(f"{review} is in Week 1 but not in the manifest, and is neither unlabelled nor a repeat")
print("Week 1 reviews left out of the dataset:")
display(pd.DataFrame(excluded, columns=["review_id", "reason"]))

# English text <-> translations
digit_map = str.maketrans("०१२३४५६७८९০১২৩৪৫৬৭৮৯", "01234567890123456789")      # Hindi/Marathi and Bengali digits -> 0-9
def numbers_in(text):
    return set(re.findall(r"\d+(?:\.\d+)?", text.translate(digit_map)))

in_manifest = set(manifest["review_id"])
order = manifest.set_index("review_id").loc[[r for r in week1[week1["language"] == "hi"]["review_id"] if r in in_manifest]]
english_numbers = [numbers_in(t) for t in order["english"]]
for language in ["hi", "bn", "mr"]:
    indic_numbers = [numbers_in(t) for t in texts.loc[order.index, language]]
    same_row = np.mean([e <= i for e, i in zip(english_numbers, indic_numbers) if e])
    next_row = np.mean([e <= i for e, i in zip(english_numbers[:-1], indic_numbers[1:]) if e])
    print(f"{language}: English numbers found in the translation: same row {same_row:.2f} | next row (should be ~0) {next_row:.2f}")
    if same_row < 0.9 or next_row > 0.2:
        raise ValueError(f"{language}: the English reviews do not line up with the {language} rows. Do not continue.")

Week 1 reviews left out of the dataset:


,review_id,reason
0,test_662,"repeated review (same text in hi, bn and mr)"
1,test_663,missing sentiment label
2,test_694,missing sentiment label
3,test_744,"repeated review (same text in hi, bn and mr)"


hi: English numbers found in the translation: same row 0.98 | next row (should be ~0) 0.01
bn: English numbers found in the translation: same row 0.97 | next row (should be ~0) 0.01
mr: English numbers found in the translation: same row 0.96 | next row (should be ~0) 0.01


In [21]:
# Joining the intent labels and class numbers to the split
reviews = manifest.merge(labels[["review_id", "intent", "unsure"]], on="review_id", how="left", validate="one_to_one")
if reviews["intent"].isna().any():
    raise ValueError("some reviews have no intent label")

# numeric class ids. Sentiment: the manifest's own label_id. Intent: numbered in the order Inquiry, Opinion, Factual;
# a class with no examples gets no id.
sentiment_pairs = reviews[["label", "label_id"]].drop_duplicates()
if len(sentiment_pairs) != reviews["label"].nunique() or len(sentiment_pairs) != reviews["label_id"].nunique():
    raise ValueError("label and label_id in the manifest do not match one-to-one: " + str(sentiment_pairs.values.tolist()))
if not set(reviews["label"]) <= {"Negative", "Neutral", "Positive"}:
    raise ValueError(f"unexpected sentiment values {sorted(set(reviews['label']))}")
sentiment_ids = dict(zip(sentiment_pairs["label"], sentiment_pairs["label_id"]))
intent_names = [name for name in allowed_intents if (reviews["intent"] == name).any()]
intent_ids = {name: i for i, name in enumerate(intent_names)}
reviews["intent_id"] = reviews["intent"].map(intent_ids)

print("sentiment classes:", sentiment_ids)
print("intent classes   :", intent_ids)
display(pd.crosstab(reviews["part"], reviews["label"]))
display(pd.crosstab(reviews["part"], reviews["intent"]))
print("Original Week 1 split (rows) against the Week 2 part (columns):")
display(pd.crosstab(reviews["split"], reviews["part"]))

sentiment classes: {'Negative': 0, 'Positive': 1}
intent classes   : {'Opinion': 0, 'Factual': 1}


label,Negative,Positive
part,,
test,86,87
train,400,406
val,86,87


intent,Factual,Opinion
part,,
test,13,160
train,66,740
val,16,157


Original Week 1 split (rows) against the Week 2 part (columns):


part,test,train,val
split,,,
test,151,700,145
validation,22,106,28


In [ ]:
# Build the dataset  and check it
rows = week1[["review_id", "language", "split", "text"]].rename(columns={"split": "source_split"})
review_columns = ["review_id", "part", "label", "label_id", "intent", "intent_id", "unsure"]

dataset = rows.merge(reviews[review_columns], on="review_id", how="inner", validate="many_to_one")
dataset = dataset.sort_values(["language", "review_id"]).reset_index(drop=True)

if len(dataset) != 3 * len(reviews):
    raise ValueError(f"Expected {3 * len(reviews)} rows, got {len(dataset)}.")
if dataset.duplicated(["review_id", "language"]).any():
    raise ValueError("Duplicate review-language rows found.")
if dataset.isna().any().any():
    raise ValueError("Missing values: " + str(dataset.isna().sum()[lambda s: s > 0].to_dict()))
if dataset.groupby("review_id")["part"].nunique().gt(1).any():
    raise ValueError("Some reviews appear in multiple partitions.")
if dataset.groupby(["language", "text"])["part"].nunique().gt(1).any():
    raise ValueError("Duplicate text leakage across partitions.")

rows_per_language = dataset.groupby(["language", "part"]).size().unstack(fill_value=0)
if not rows_per_language.eq(rows_per_language.iloc[0]).all().all():
    raise ValueError("Language row counts differ across partitions.")

# the sentiment label of every row must equal the Week 1 label of the same review and language
dataset_label = dataset.set_index(["review_id", "language"])["label"]
if (dataset_label != week1.set_index(["review_id", "language"])["label"].reindex(dataset_label.index)).any():
    raise ValueError("the sentiment label of some rows differs from the Week 1 file")

print("Dataset validation passed.")
display(rows_per_language)

Dataset validation passed.


part,test,train,val
language,,,
bn,173,806,173
hi,173,806,173
mr,173,806,173


In [22]:
# Saved the prepared dataset
dataset.to_csv(DATASET_PATH, index=False, encoding="utf-8")

print(f"Saved: {DATASET_PATH.name} | Rows: {len(dataset)} | Unique reviews: {dataset['review_id'].nunique()}")
print("Columns:", list(dataset.columns))
print("Sentiment classes:", list(sentiment_ids), "| Intent classes:", intent_names)
if "Neutral" not in sentiment_ids:
    print("  - no 'Neutral' examples exist, so the model cannot learn or be tested on it")
if "Inquiry" not in intent_names:
    print("  - no 'Inquiry' examples exist, so the model cannot learn or be tested on it")
print("Note: the intent labels in intent_labels.csv are silver labels (not human-verified ground truth).")

Saved: week2_dataset.csv | Rows: 3456 | Unique reviews: 1152
Columns: ['review_id', 'language', 'source_split', 'text', 'part', 'label', 'label_id', 'intent', 'intent_id', 'unsure']
Sentiment classes: ['Negative', 'Positive'] | Intent classes: ['Opinion', 'Factual']
  - no 'Neutral' examples exist, so the model cannot learn or be tested on it
  - no 'Inquiry' examples exist, so the model cannot learn or be tested on it
Note: the intent labels in intent_labels.csv are silver labels (not human-verified ground truth).
